# Preamble

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import warnings
import keysight.qcs as qcs
import numpy as np
import matplotlib.pyplot as plt
# import sys
# sys.path.append("..")

from keysight.qcs.experiments import ResonatorSpectroscopy, ResonatorSpectroscopy2D, DispersiveShift
from keysight.qcs.experiments import QubitSpectroscopy,RabiExperiment,T1Experiment,RamseyExperiment,IQDistributionExperiment
from keysight.qcs.analysis import FanoResonance, ComplexLorentzian

from preamble.pump_utils import *
from preamble.mapper_utils import *
from preamble.calibration_utils import *
from preamble.pca_gmm_utils import *
from preamble.fitter import *


us,ns,MHz,GHz=1e-6,1e-9,1e6,1e9
pi=np.pi

In [ ]:
cal_file='./config/260702_cal.json'
# set the qubit topology
qubits = qcs.Qudits([0,1,2])
n_qubits=len(qubits)

pairs = [(qubits[i], qubits[i + 1]) for i in range(n_qubits - 1)]
n_mq = len(pairs)

topology = qcs.QuditGraph(qubits, pairs)

# set the local oscillator frequency
LO_freq_readout =  6.1*GHz
LO_freq_qubit = 4.3*GHz

# load the calibration variables
variables_from_json = load_json(cal_file)
variables=make_qcs_var(
            variables_from_json=variables_from_json,
            qubits=qubits,
        )

mapper = qcs.ChannelMapper('localhost')
address = load_address("./config/channel_address.yaml")

channel_mapping(mapper,qubits,address)
set_lo_frequencies_RO(mapper,qubits,address,LO_freq_readout)
set_lo_frequencies_XY(mapper,qubits,address,LO_freq_qubit) 
# set_lo_frequencies_XY(mapper,qubits[0],address,4.1*GHz)

backend = qcs.HclBackend(mapper, fpga_postprocessing=True ,init_time=150*us)

cals = MyCalibrationSet(
        topology=topology,
        channels=mapper.channels,
        variables=variables)
cals.export_values(cal_file)

# shorten the variables name
c=cals.variables


print("Backend is ready:", backend.is_system_ready())


In [ ]:
# set the port of windfreak
# port='socket://163.152.38.107:4000' 
port='COM3'
pump = PumpController(port=port).connect()
pump.set(ch=1,freq=7.52*GHz, power=8.4)
pump.close_all()


# Abort program 

In [ ]:
current_id=backend.get_program_execution_history()[0]['accession_id']
backend.abort_program(current_id)

# Q0

## Amplitude

In [ ]:
def error_amp_program(n_reps, n_shots=1000):
    error_amp = qcs.Program()
    error_amp.add_measurement(target_qubit, new_layer=True)
    
    error_amp.add_gate(qcs.GATES.x90, target_qubit,pre_delay=3*us)
    for k in range(n_reps-1):
        error_amp.add_gate(qcs.GATES.x90, target_qubit)
    error_amp.add_measurement(target_qubit, new_layer=True)
    error_amp = qcs.LinkerPass(*cals.linkers.values()).apply(error_amp)

    error_amp.n_shots(n_shots)

    return error_amp

In [ ]:
target_qubit=qubits[0]

# Set the frequency range for sweeping
amp_center = c.sx_amp[target_qubit.labels].value
amp_range = 0.1
n_steps = 11
n_reps=2

amplitudes=np.linspace(amp_center-amp_range/2, amp_center+amp_range/2, n_steps)
amplitudes=qcs.Array('amplitudes',value=amplitudes, dtype=float)

##############################
error_amp = error_amp_program(n_reps=n_reps,n_shots=4000)

error_amp.sweep(amplitudes, c.sx_amp[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
error_amp = qcs.Executor(backend).execute(error_amp)
pump.off(ch=1).close_all()

In [ ]:
z_exps=[]
for k in range(len(amplitudes.value)):
    init_idx=np.where(error_amp.get_classified_array(acq_index=0)[0,k]==0)
    counts=np.unique(error_amp.get_classified_array()[0,k][init_idx],return_counts=True)[1]
    z_exp=(counts[0]-counts[1])/(counts[0]+counts[1])
    z_exps.append(z_exp)

plt.plot(amplitudes.value,z_exps)

## Amplitude fine

In [ ]:
target_qubit=qubits[0]

# Set the frequency range for sweeping
amp_center = c.sx_amp[target_qubit.labels].value
amp_range = 0.01
n_steps = 11
n_reps=42

amplitudes=np.linspace(amp_center-amp_range/2, amp_center+amp_range/2, n_steps)
amplitudes=qcs.Array('amplitudes',value=amplitudes, dtype=float)

##############################
error_amp = error_amp_program(n_reps=n_reps,n_shots=4000)

error_amp.sweep(amplitudes, c.sx_amp[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
error_amp = qcs.Executor(backend).execute(error_amp)
pump.off(ch=1).close_all()

In [ ]:
z_exps=[]
for k in range(len(amplitudes.value)):
    init_idx=np.where(error_amp.get_classified_array(acq_index=0)[0,k]==0)
    counts=np.unique(error_amp.get_classified_array()[0,k][init_idx],return_counts=True)[1]
    z_exp=(counts[0]-counts[1])/(counts[0]+counts[1])
    z_exps.append(z_exp)

plt.plot(amplitudes.value,z_exps)

# Make cal_backup

In [ ]:
cals.export_values(cal_file.removesuffix('.json')+'_backup.json')